# 10 — ARIMA Baseline

Stage 4a, classical track. Establishes three reference baselines (naive, seasonal-naive, the dataset's own `Demand Forecast` column) and a per-series SARIMA model, all evaluated **one-step-ahead with true history** over the test window (2023-10-01 to 2024-01-01) — the same protocol the LSTM in `11_lstm_forecasting.ipynb` uses, so the eventual comparison in `12_forecast_comparison.ipynb` is apples-to-apples.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.data.load import read_features
from src.forecasting.splits import time_split
from src.forecasting.evaluate import evaluate_forecasts, compute_scales
from src.forecasting.arima_model import run_arima_all_series
from src.forecasting.backtest import backtest_summary

pd.set_option("display.max_columns", 50)

features = read_features()
train, val, test = time_split(features)
print(f"train={len(train)} val={len(val)} test={len(test)}  series={train.groupby(config.ID_COLS, observed=True).ngroups}")


train=5460 val=920 test=930  series=10


## Shared MASE scale

Computed once on TRAIN only, per series, and reused by every model's MASE so all comparisons use the same yardstick.

In [2]:
scales = compute_scales(train)
pd.Series(scales).describe()


count     10.000000
mean     125.823933
std        5.378641
min      117.100186
25%      123.276902
50%      126.551020
75%      129.403989
max      133.217069
dtype: float64

## Baselines

`features_full.parquet` already carries leakage-safe `units_sold_lag_1` and `units_sold_lag_7` columns (Stage 3), so the naive and seasonal-naive baselines are just those columns read off the test split directly — no extra computation needed. The dataset's own `Demand Forecast` column is the third baseline.

In [3]:
def make_baseline(test_df, pred_col, model_name):
    d = test_df[config.ID_COLS + [config.DATE_COL, config.TARGET_COL, pred_col]].copy()
    d = d.rename(columns={config.TARGET_COL: "y_true", pred_col: "y_pred"})
    d["model"] = model_name
    return d.dropna(subset=["y_pred"])

baseline_naive = make_baseline(test, "units_sold_lag_1", "Naive")
baseline_seasonal_naive = make_baseline(test, "units_sold_lag_7", "SeasonalNaive")
baseline_dataset = make_baseline(test, "Demand Forecast", "DatasetForecast")

baselines = {
    "Naive": baseline_naive,
    "SeasonalNaive": baseline_seasonal_naive,
    "DatasetForecast": baseline_dataset,
}

baseline_rows = []
for name, d in baselines.items():
    overall = evaluate_forecasts(d, scales)["overall"]
    baseline_rows.append({"model": name, **overall.to_dict()})

baseline_summary = pd.DataFrame(baseline_rows)
baseline_summary


,model,MAE,RMSE,sMAPE,MASE_macro,MASE_volume_weighted
0,Naive,118.233333,151.629624,89.856236,0.940797,0.944370
1,SeasonalNaive,122.484946,154.244322,92.449877,0.977751,0.982678
2,DatasetForecast,8.346806,10.070042,15.532365,0.066431,0.066513


## SARIMA per series

`pmdarima.auto_arima(seasonal=True, m=7)` fit once per series on train, then walked forward through the test window one day at a time: predict 1 step, then `.update()` with that day's true value before predicting the next — same evaluation protocol the LSTM uses. Every series' run (chosen order, seasonal_order, metrics) is logged to MLflow under experiment `forecasting_arima`.

In [4]:
arima_forecasts, arima_orders = run_arima_all_series(train, test, log_to_mlflow=True)
arima_orders


2026/08/07 17:10:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/07 17:10:48 INFO mlflow.store.db.utils: Updating database tables


2026/08/07 17:10:48 INFO mlflow.tracking.fluent: Experiment with name 'forecasting_arima' does not exist. Creating a new experiment.


,Store ID,Product ID,order,seasonal_order,MAE,RMSE,MASE
0,S002,P0001,"(0, 0, 0)","(0, 0, 0, 7)",89.001315,108.486612,0.686526
1,S002,P0009,"(3, 0, 3)","(1, 0, 1, 7)",97.453717,118.892400,0.792833
2,S002,P0020,"(0, 0, 1)","(0, 0, 0, 7)",92.453794,112.368014,0.718391
3,S003,P0005,"(0, 0, 0)","(0, 0, 0, 7)",84.407331,98.957426,0.678775
4,S003,P0013,"(0, 0, 0)","(0, 0, 0, 7)",84.569203,101.824204,0.657694
5,S003,P0014,"(0, 0, 0)","(1, 0, 0, 7)",96.455712,119.135412,0.735834
6,S003,P0017,"(0, 0, 0)","(0, 0, 0, 7)",91.805464,115.307935,0.783991
7,S004,P0016,"(0, 0, 0)","(0, 0, 0, 7)",79.735161,95.536239,0.640352
8,S005,P0003,"(0, 0, 0)","(0, 0, 0, 7)",98.848581,121.359280,0.836779
9,S005,P0015,"(0, 0, 0)","(0, 0, 0, 7)",91.547538,108.450228,0.687206


In [5]:
arima_overall = evaluate_forecasts(arima_forecasts, scales)["overall"]
arima_overall


MAE                      90.627781
RMSE                    110.362097
sMAPE                    69.952217
MASE_macro                0.721838
MASE_volume_weighted      0.724424
dtype: float64

## Baselines vs SARIMA — comparison so far

In [6]:
comparison = pd.concat(
    [baseline_summary, pd.DataFrame([{"model": "SARIMA", **arima_overall.to_dict()}])],
    ignore_index=True,
)
comparison = comparison[["model", "MAE", "RMSE", "sMAPE", "MASE_macro", "MASE_volume_weighted"]]
comparison.sort_values("MASE_macro")


,model,MAE,RMSE,sMAPE,MASE_macro,MASE_volume_weighted
2,DatasetForecast,8.346806,10.070042,15.532365,0.066431,0.066513
3,SARIMA,90.627781,110.362097,69.952217,0.721838,0.724424
0,Naive,118.233333,151.629624,89.856236,0.940797,0.944370
1,SeasonalNaive,122.484946,154.244322,92.449877,0.977751,0.982678


## Backtest robustness check

Splits the already-computed one-step-ahead test forecasts into 2 contiguous chronological chunks and scores each separately, to see whether SARIMA's error is stable across the test window rather than concentrated in one sub-period.

In [7]:
backtest_summary(arima_forecasts, scales, n_origins=2)


,origin,start_date,end_date,MAE,RMSE,sMAPE,MASE_macro
0,1,2023-10-01,2023-11-16,92.659125,112.673145,70.731867,0.737907
1,2,2023-11-17,2024-01-01,88.552278,107.949720,69.155618,0.705419


## Save outputs

In [8]:
config.FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
arima_forecasts.to_parquet(config.FORECASTS_DIR / "arima_forecasts.parquet", index=False)

baselines_all = pd.concat(baselines.values(), ignore_index=True)
baselines_all.to_parquet(config.FORECASTS_DIR / "baseline_forecasts.parquet", index=False)

config.ARIMA_DIR.mkdir(parents=True, exist_ok=True)
arima_orders.to_csv(config.ARIMA_DIR / "orders.csv", index=False)

print("saved arima_forecasts.parquet, baseline_forecasts.parquet, models/arima/orders.csv")


saved arima_forecasts.parquet, baseline_forecasts.parquet, models/arima/orders.csv


## Summary

- Three baselines established (Naive, SeasonalNaive, the dataset's own DatasetForecast) plus per-series SARIMA, all scored with the same MASE yardstick (seasonal-naive scale computed on train only).
- SARIMA orders chosen per series via `auto_arima`, logged individually to MLflow (`forecasting_arima` experiment).
- Backtest chunking shows whether SARIMA's error is stable across the test window.
- Forecast outputs saved for the final comparison in `12_forecast_comparison.ipynb`, once `11_lstm_forecasting.ipynb` has also produced its forecasts.